# LLMC GSM8K Results Analysis
分析 results_llmc 資料夾中各模型在 GSM8K 數據集上的表現

In [10]:
import json
import os
import pandas as pd
from pathlib import Path

In [11]:
# 定義 results_llmc 資料夾路徑
results_dir = Path('../results_llmc')

# 收集所有模型的 GSM8K 結果
data = []

for model_dir in sorted(results_dir.iterdir()):
    if model_dir.is_dir():
        gsm8k_file = model_dir / 'gsm8k_results.json'
        
        if gsm8k_file.exists():
            with open(gsm8k_file, 'r') as f:
                results = json.load(f)
            
            # 提取模型名稱和量化方法
            model_name = model_dir.name
            
            # 從模型名稱中提取量化方法
            if '-llmc-' in model_name:
                method = model_name.split('-llmc-')[1].split('-')[0]
                if 'sparsegpt' in model_name:
                    # 處理 sparsegpt 的特殊情況
                    if '2x4' in model_name:
                        method = 'sparsegpt-2x4'
                    elif 'sparse30' in model_name:
                        method = 'sparsegpt-sparse30'
            else:
                method = 'unknown'
            
            # 提取關鍵指標
            data.append({
                'Model': model_name,
                'Method': method.upper(),
                'Accuracy (%)': round(results['accuracy'] * 100, 2),
                'Correct': results['correct'],
                'Total': results['total'],
                'GPU Peak (MB)': results.get('gpu_peak_mb', 'N/A'),
                'Time (s)': round(results['total_generation_time_sec'], 2),
                'Throughput (tokens/s)': round(results['throughput_tokens_per_sec'], 2)
            })

# 創建 DataFrame
df = pd.DataFrame(data)
print(f"共找到 {len(df)} 個模型的 GSM8K 結果")

共找到 5 個模型的 GSM8K 結果


In [12]:
# 顯示完整表格
print("\n=== GSM8K Results - All Models ===")
print(df.to_string(index=False))


=== GSM8K Results - All Models ===
                                                        Model             Method  Accuracy (%)  Correct  Total  GPU Peak (MB)  Time (s)  Throughput (tokens/s)
   Llama-3.2-1B-Instruct-llmc-autoround-W4A16-20260204_000527          AUTOROUND         21.46      283   1319            NaN    295.73                5028.83
         Llama-3.2-1B-Instruct-llmc-awq-W4A16-20260131_121301                AWQ         22.82      301   1319            NaN    196.22                7633.68
        Llama-3.2-1B-Instruct-llmc-gptq-W4A16-20260130_122554               GPTQ         30.63      404   1319            NaN    235.10                6422.07
     Llama-3.2-1B-Instruct-llmc-sparsegpt-2x4-20260204_015554      SPARSEGPT-2X4          2.05       27   1319      2983.4961    393.81                3721.49
Llama-3.2-1B-Instruct-llmc-sparsegpt-sparse30-20260203_202002 SPARSEGPT-SPARSE30         26.46      349   1319      2478.6914    484.93                3063.10


In [13]:
# 顯示簡化表格（只包含主要指標）
df_simple = df[['Method', 'Accuracy (%)', 'GPU Peak (MB)', 'Time (s)']].copy()
print("\n=== GSM8K Results - Simplified ===")
print(df_simple.to_string(index=False))


=== GSM8K Results - Simplified ===
            Method  Accuracy (%)  GPU Peak (MB)  Time (s)
         AUTOROUND         21.46            NaN    295.73
               AWQ         22.82            NaN    196.22
              GPTQ         30.63            NaN    235.10
     SPARSEGPT-2X4          2.05      2983.4961    393.81
SPARSEGPT-SPARSE30         26.46      2478.6914    484.93


In [14]:
# 按準確率排序
df_sorted = df.sort_values('Accuracy (%)', ascending=False)
print("\n=== GSM8K Results - Sorted by Accuracy ===")
print(df_sorted[['Method', 'Accuracy (%)', 'GPU Peak (MB)', 'Time (s)']].to_string(index=False))


=== GSM8K Results - Sorted by Accuracy ===
            Method  Accuracy (%)  GPU Peak (MB)  Time (s)
              GPTQ         30.63            NaN    235.10
SPARSEGPT-SPARSE30         26.46      2478.6914    484.93
               AWQ         22.82            NaN    196.22
         AUTOROUND         21.46            NaN    295.73
     SPARSEGPT-2X4          2.05      2983.4961    393.81


In [16]:
# 生成 Markdown 格式的表格
print("\n=== Markdown Format ===")
print("\n| Method | Accuracy (%) | GPU Peak (MB) | Time (s) |")
print("|--------|-------------|---------------|----------|")

for _, row in df_simple.iterrows():
    gpu = row['GPU Peak (MB)'] if row['GPU Peak (MB)'] != 'N/A' else 'N/A'
    print(f"| {row['Method']} | {row['Accuracy (%)']} | {gpu} | {row['Time (s)']} |")


=== Markdown Format ===

| Method | Accuracy (%) | GPU Peak (MB) | Time (s) |
|--------|-------------|---------------|----------|
| AUTOROUND | 21.46 | nan | 295.73 |
| AWQ | 22.82 | nan | 196.22 |
| GPTQ | 30.63 | nan | 235.1 |
| SPARSEGPT-2X4 | 2.05 | 2983.4961 | 393.81 |
| SPARSEGPT-SPARSE30 | 26.46 | 2478.6914 | 484.93 |


In [ ]:
# 統計摘要
print("\n=== Summary Statistics ===")
print(f"Best Accuracy: {df_sorted.iloc[0]['Method']} - {df_sorted.iloc[0]['Accuracy (%)']}%")
print(f"Fastest: {df.loc[df['Time (s)'].idxmin()]['Method']} - {df['Time (s)'].min():.2f}s")
print(f"Highest Throughput: {df.loc[df['Throughput (tokens/s)'].idxmax()]['Method']} - {df['Throughput (tokens/s)'].max():.2f} tokens/s")


=== Summary Statistics ===
Best Accuracy: GPTQ - 30.63%
Fastest: AWQ - 196.22s
Highest Throughput: AWQ - 7633.68 tokens/s


In [ ]:
# 保存為 CSV
output_file = 'llmc_gsm8k_results.csv'
df.to_csv(output_file, index=False)
print(f"\n結果已保存至: {output_file}")


結果已保存至: llmc_gsm8k_results.csv
